<a href="https://colab.research.google.com/github/rsoumyadeep/andrej-karpathy-skills/blob/main/Assignment_3_for_DS_207_Intro_to_NLP_RL_Methods_for_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS-207: Introduction to NLP Assignment 3 - RL Methods for NLP

TAs:
- Purva Parmar [purvaparmar@iisc.ac.in]
- Victor Azad [victorazad@iisc.ac.in]


In this assignment, we will look at a small language model, under 100M parameter scale, and attempt to train the model to do simple mathematical word-problems. We start with Supervised Fine-Tuning (SFT) to establish a chain-of-thought reasoning, then use Direct Preference Optimization (DPO) to align the model with preference pairs. We also optionally take a look at Group Relative Policy Optimization (GRPO), which works with multiple answers in a group and uses rewards without a separate critic model.

## Guidelines for Attempting the Assignment


1. Please read the function docs and comments carefully. They provide specific instructions and examples for implementing each function. Follow these instructions precisely - neither oversimplify nor overcomplicate your implementations. Deviating from the provided implementation guidelines may result in lost marks.

2. Write your logic in the cells which have the comment `# ADD YOUR CODE HERE`, between the `# BEGIN CODE` and `# END CODE` comments. These cells are also demarcated by the special start (`# ==== BEGIN EVALUATION PORTION`) and end (`# ==== END EVALUATION PORTION`) comments.

3. Do **NOT** remove any of these comments from the designated cells, otherwise your assignment may not be evaluated correctly.

4. All imports that should be necessary are already provided as part of the notebook. You are not allowed to add any additional imports.

5. Do not modify anything in the cells which start with `# Please do not change anything in the following cell`.

6. Ensure you only add code in designated areas, otherwise your assignment will not be evaluated. If you encounter any errors in the supporting cells during execution, contact the respective TAs.

7. Ensure that the total runtime of the assignment is less than 100 minutes (excluding GRPO). Exceeding this time may result in deductions of marks.

8. **Important**: Use of AI-assistive technologies such as ChatGPT or GitHub CoPilot is not permitted for this assignment. Ensure that all attempts are solely your own. Not following this rule can incur heavy penalty, including getting NO GRADE for this assignment.


### Submission Instructions

1. When you have completely attempted the assignment, **export the current notebook as a `.py` file**, and rename it with the following name: `SAPName_SRNo_assignment3.py`, where `SAPName` would be your name as per SAP record, and `SRNo` will be the last 5 digits of your IISc SR number. For example, IISc student with SAP name Twyla Linda (SR no - 04-03-00-10-22-20-1-15329) would use `Twyla_Linda_15329_assignment3.py`.

2. By the end of this assignment, you will also have an `sft_model/`, a `dpo_model/` and optionally a `grpo_model/`, each containing five files, as can be checked in the Colab filesystem:
    - `model.safetensors` - The model weights, this can be about 300 MB for each model.
    - `tokenizer_config.json`
    - `tokenizer.json`
    - `generation_config.json`
    - `config.json`

    We provide small utility code to zip these folders at the end of the notebook. After running them, you can directly download the `sft_model.zip`, `dpo_model.zip` and optionally the `grpo_model.zip` files. Each of these should be roughly 300 MB.

3. You should have four files ready for submission.

    ```python
    ├── SAPName_SRNo_assignment3.py
    ├── dpo_model.zip
    ├── grpo_model.zip  # Optional
    └── sft_model.zip
    ```

4. Upload each of these four files individually to the Assignment 3 on Teams.

5. Do NOT upload the `data/` folder as part of the submission.


**If you have any confusion regarding submission instructions, please ask the respective TAs.**

## Marks Distribution

- Supervised Fine-Tuning (SFT): 40 marks
- Direct Preference Optimization (DPO): 60 marks
- Group Relative Policy Optimization (GRPO): 0 marks [GRPO is an Optional, Extra Section]

## Setup

Arya Kapoor is three weeks into her first job.

Fresh out of her master's program, she has just joined a small edtech startup that creates AI solutions in the education industry.

Arya's job title is AI Engineer. Her first project: build a model that can solve
grade-school word problems the way a patient tutor would, not just spitting out a number,
but showing its work, step by step, so a child can follow along and actually learn.

### Student Details

In [ ]:
# Add your info here
SAPName = "Arya Kapoor"                # Add here your name as per SAP
SRNo = "24013"                         # Add here last 5 digits of your IISc Sr Number

### Library Imports

In [ ]:
# Do not change anything in this cell block

import gc
import json
import logging
import math
import os
import re
import shutil

import gdown
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Suppressing logging
logging.getLogger("datasets").setLevel(logging.WARNING)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

### Datasets Download

Arya's first question is: *what data should the model learn from?*

She needs a collection of grade-school math word problems, problems of the kind
a 3rd or 4th grader would find in a textbook. Crucially, she doesn't just want
problems and bare answers. She wants problems paired with *step-by-step solutions*,
so the model can learn the reasoning process and not just memorise answers.

Her search leads her to **GSM-8K** (Grade School Math - Approximately 8,500 problems), a dataset
created by OpenAI researchers in 2021. Here is what makes it special:

- It contains **~8,500 linguistically diverse word problems** written at a
  grade-school level (roughly US grades 1-8). Problems involve a few reasoning
  steps and require basic arithmetic: addition, subtraction, multiplication,
  division.

- Every problem comes with a **chain-of-thought solution**: a natural-language
  walkthrough of *how* to arrive at the answer, not just the answer itself.
  This is exactly what Arya needs if she wants the model to "show its work."

- Each answer ends with a special delimiter `####` followed by the final
  numerical answer. For example:
    - "Johnny has 5 apples and eats 2. How many are left?",
    - "Answer: We start with 5 apples. Johnny eats 2, so 5 - 2 = 3. #### 3"

- GSM-8K became a *standard benchmark* for evaluating how well language models
  reason. You will often see papers reporting a model's "GSM-8K accuracy" as a
  proxy for mathematical reasoning ability. Models like GPT-4 score >95% —
  but tiny models of the size MathMind can afford to deploy score much lower,
  which is exactly why Arya's work is interesting.

The datasets Arya downloads below are GSM-8K-style, of a similar format and style, created
for the purpose of this task.

The dataset files are stored in **JSONL format** (JSON Lines) - one JSON object
per line.

In [ ]:
# Do not change anything in this cell block

# Download datasets
# In case some there was an error downloading some file, run this cell again

def setup_data_directory(directory="data"):
    """Creates the data directory if it doesn't already exist."""
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created directory: {directory}")

def download_dataset(file_id, output_path, overwrite=False):
    """Downloads a file from Google Drive only if it doesn't exist or overwrite is True."""
    if os.path.exists(output_path) and not overwrite:
        print(f"File already exists: {output_path}. Skipping download.")
        return

    url = f'https://drive.google.com/uc?id={file_id}'
    try:
        gdown.download(url, output_path, quiet=False)
    except Exception as e:
        print(f"Failed to download {output_path}: {e}")

def initialize_project_data(force_reload=False):
    """Main function to setup folders and download all datasets."""
    setup_data_directory("data")

    datasets = {
        "data/train.jsonl": "1M1DEfy6CVPx8HU4abzT71zyrUfePRXw0",
        "data/eval.jsonl": "1jjWJ3x7AjLK5Q35LIQAheRO32d-KZz3D",
        "data/dpo_train.jsonl": "16vEtdw_fO_3TyVzVg0uey-HH2teujEsw",
        "data/dpo_eval.jsonl": "1cxUpymIRHux4oFUt3qBK9lFj8bQc4-8u",
        "data/grpo_train.jsonl": "1YSdeXppWrkFeKZwKLWTdKAPoj4huhBjb",
        "data/grpo_eval.jsonl": "10WzbpEVRHtxzniLwSgoyIlytk-BrgWol"
    }

    for path, file_id in datasets.items():
        download_dataset(file_id, path, overwrite=force_reload)

# Run the initialization
# In case some there was an error downloading some file, run this cell again
initialize_project_data(force_reload=False)

In [ ]:
# Do not change anything in this cell block

# GSM8K Dataset
# The dataset is formatted as a list of dictionaries (i.e. [{}, {}]) and each item has three keys "question", "reasoning" and, "answer".

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

# For training, Arya needs these to be separate fields. The model will be trained
# to produce the reasoning first, and then the answer, and the training loss will
# be computed on both. Merging them into one blob makes that harder.
def transform_gsm8k(item: dict) -> dict:
    """
    Transforms a raw GSM8K dataset item by splitting the answer into reasoning steps and the final answer.

    Args:
        item (dict): A dictionary from the GSM8K dataset containing 'question' and 'answer' keys.
                     The 'answer' string contains the chain of thought and the final answer
                     separated by '####'.

    Returns:
        dict: A parsed dictionary containing 'question', 'reasoning', and 'answer'.

    Note:
        Example Input:
            {'question': 'If Johnny has 5 apples and eats 2, how many are left?',
             'answer': 'First subtract 2 from 5. #### 3'}
        Example Output:
            {'question': 'If Johnny has 5 apples and eats 2, how many are left?',
             'reasoning': 'First subtract 2 from 5.',
             'answer': '3'}
    """
    reasoning, answer = item["answer"].split("####")

    return {
        "question": item["question"],
        "reasoning": reasoning.strip(),
        "answer": answer.strip()
    }

train_file = "./data/train.jsonl"
eval_file = "./data/eval.jsonl"

train_raw = load_jsonl(train_file)
eval_raw = load_jsonl(eval_file)

train_data = [transform_gsm8k(item) for item in train_raw]
eval_data  = [transform_gsm8k(item) for item in eval_raw]

sample_data = train_data[:2]

print("Dataset Details\n")
print(f"Train samples: {len(train_data)}")
print(f"Eval samples: {len(eval_data)}")

print("Sample Data\n")
print(f"Question: \n{sample_data[0]['question']}\n")
print(f"Reasoning:\n{sample_data[0]['reasoning']}\n")
print(f"Answer: {sample_data[0]['answer']}")

## Base Model and Evaluation Functions

Arya can't train a language model from scratch. That would take weeks and
thousands of GPU-hours. Instead, she starts from a pre-trained model:
`deadMarkov/distilgpt2-math`, a compact (~82M parameter) version of GPT-2 that
has already been pre-trained by someone named Victor and Purva on text containing
mathematical content.

Think of it as a tutor who has read a lot of math textbooks but has never
actually sat down with a student and worked through problems. The pre-training
gives the model language fluency and some familiarity with arithmetic notation.
Arya's job, through fine-tuning, is to turn this bookish reader into an
effective, step-by-step problem-solver.

Mathematically, Supervised Fine-Tuning is just training the model on a dataset
containing prompt and ideal response pairs, with a lower learning rate and for
lesser time.

### Helper Functions

In [ ]:
# Do not change anything in this cell block

def extract_answer(text: str):
    """
    Extracts the final numerical answer from a generated text string.

    It first attempts to find the answer using a set of common formatting patterns
    (e.g., '#### [answer]', '\boxed{[answer]}', 'Final Answer: [answer]').
    If no specific pattern matches, it falls back to extracting the last numerical
    value found in the text.

    Args:
        text (str): The raw text generated by the model.

    Returns:
        str or None: The extracted answer string if found, otherwise None.
    """
    patterns = [
        r"####*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
        r"\\boxed\{\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)\s*\}",
        r"[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
        r"[Aa]nswer\s*[:\-]?\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
        r"[Tt]he answer is\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
        r"=\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)\s*$"
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    # Fallback: find all numbers and return the last one
    numbers = re.findall(r"-?\d+(?:\.\d+)?(?:e[-+]?\d+)?", text)
    if numbers:
        return numbers[-1]

    return None

In [ ]:
# Do not change anything in this cell block

def flush_memory(*objects):
    for obj in objects:
        # Release heavy sub-objects through the shared reference
        for attr in ("model", "optimizer", "scheduler", "train_loader"):
            if hasattr(obj, attr):
                setattr(obj, attr, None)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Memory flushed successfully.")

### Base Model

In [ ]:
# Base Model Class. Do not change anything here.

class BaseModel:
    """
    A wrapper class for initializing and managing a Hugging Face Causal Language Model and its tokenizer.
    """
    def __init__(self, model_name: str = "deadMarkov/distilgpt2-math", device: str = None):
        """
        Initializes the model and tokenizer from the specified Hugging Face model hub name.

        Args:
            model_name (str): The Hugging Face model identifier.
            device (str, optional): The device to load the model on ('cpu' or 'cuda').
        """
        if device:
            self.device = device
        else:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)

        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.config.pad_token_id = self.tokenizer.pad_token_id

    @property
    def vocab_size(self) -> int:
        """
        Retrieves the vocabulary size of the loaded model.

        Returns:
            int: The number of tokens in the model's vocabulary.
        """
        size = self.model.config.vocab_size
        return size

    @property
    def param_count(self) -> int:
        """
        Calculates the total number of parameters in the model.

        Returns:
            int: Total parameter count.
        """
        num_params = sum(p.numel() for p in self.model.parameters())
        return num_params

    def load(self, load_dir: str):
        """
        Loads a pre-trained model and tokenizer from a local directory.

        Args:
            load_dir (str): Path to the directory containing saved model files.
        """
        self.model = AutoModelForCausalLM.from_pretrained(load_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(load_dir)
        self.model.to(self.device)

    def save(self, save_dir: str):
        """
        Saves the current model and tokenizer to a specified local directory.

        Args:
            save_dir (str): The directory path to save the files. Creates it if it doesn't exist.
        """
        os.makedirs(save_dir, exist_ok=True)
        self.model.save_pretrained(save_dir)
        self.tokenizer.save_pretrained(save_dir)

    def generate(self, input_text, max_new_tokens=256, temperature=0.7, top_p=0.95, do_sample=True, repetition_penalty=1.2):
        self.model.eval()
        device = next(self.model.parameters()).device
        inputs = self.tokenizer(input_text, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = self.model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature,
                        top_p=top_p,
                        do_sample=do_sample,
                        repetition_penalty=repetition_penalty
                      )

        output_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        return output_text

In [ ]:
# Do not change anything in this cell block

base_model = BaseModel()
print("\nBase Model Details\n")
print(f"Model Name: {base_model.model.name_or_path}")
print(f"Hidden size: {base_model.model.config.hidden_size}")
print(f"Layers: {base_model.model.config.num_hidden_layers}")
print(f"Heads: {base_model.model.config.num_attention_heads}")
print(f"Parameters: {base_model.param_count:,}")
print(f"Vocab Size: {base_model.vocab_size:,}")

In [ ]:
# Free up memory
flush_memory(base_model)

del base_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Pass@1 Evaluation

Before Arya can improve her model, she needs a way to measure improvement.
How do you score an AI tutor on a math test? She settles on two metrics: Pass@1 and Pass@K

Pass@1 measures the accuracy of a model when it is allowed a single attempt to generate the correct answer for each problem.

$$
    \text{Pass@1}=\frac{1}{N}\sum_{i=1}^{N}\mathbb{1}(\hat{y}_i=y_i)
$$

Where:
* $N$ is the total number of evaluation problems.
* $\hat{y}_i$ is the predicted answer for the $i$-th problem.
* $y_i$ is the ground truth (gold) answer for the $i$-th problem.
* $\mathbb{1}(\cdot)$ is the indicator function, evaluating to $1$ if the prediction matches the ground truth and $0$ otherwise.

This is the simplest possible metric: give the model each problem once, let it
generate one answer, and check whether that answer matches the gold answer.
Pass@1 is essentially *accuracy*. It answers the question a teacher would ask:
"If a student attempts every problem once, what fraction do they get right?"

For a model tutoring a child in real time, Pass@1 is the most important metric:
the student is waiting for an answer right now, and a wrong answer without
retries could confuse them.

In [ ]:
# Do not change anything in this cell block

def pass_at_1(model, dataset, max_new_tokens=256, temperature=0.7, top_p=0.95, do_sample=True, repetition_penalty=1.2):
    """
    Evaluates the Pass@1 metric for a given model and dataset.

    Generates exactly one response per problem. If the extracted answer from
    that single response matches the gold answer, it is counted as correct.

    Args:
        model (BaseModel): The loaded model wrapper to evaluate.
        dataset (list): A list of dictionaries containing 'question' and 'answer'.
        max_new_tokens (int): The maximum length of the generated response. Defaults to 256.
        temperature (float): Sampling temperature. Defaults to 0.7.
        top_p (float): Nucleus sampling probability. Defaults to 0.95.
        do_sample (bool): Whether to use sampling for generation. Defaults to True.
        repetition_penalty (float): Penalty for repeating tokens. Defaults to 1.2.

    Returns:
        float: The Pass@1 accuracy as a decimal between 0.0 and 1.0.
    """
    correct = 0

    for example in tqdm(dataset, desc="Evaluating pass@1"):
        question = example["question"]
        gold = example["answer"]

        pred_text = model.generate(
            input_text=question,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            repetition_penalty=repetition_penalty
        )

        pred = extract_answer(pred_text)

        if pred == gold:
            correct += 1

    return correct / len(dataset)

### Pass@K Evaluation

Pass@K measures the probability that at least one out of $k$ generated samples is correct. It is standard practice for evaluating LLMs on reasoning and coding tasks.

If we sequentially generate up to $k$ samples (as implemented below), the empirical Pass@k formula is simply the fraction of problems where at least one sample is correct:

$$
    \text{Pass@}k=\frac{1}{N}\sum_{i=1}^{N}\mathbb{1}\left(\bigvee_{j=1}^{k}(\hat{y}_{i,j}=y_i)\right)
$$

Because the model uses sampling (i.e. it's slightly random), different runs on
the same problem can produce different answers. Pass@K asks: *"If we give the
model K chances, does at least one attempt land the correct answer?"*

For Arya's purposes, Pass@K is a measure of the model's knowledge ceiling.
Does it know how to solve the problem at all, even if it isn't always consistent?
A model with high Pass@K but low Pass@1 is unreliable but knowledgeable. That's
a useful signal: maybe just more alignment training (DPO/GRPO) would help
it be more consistently correct.

**Note on Unbiased Estimators:**
In large-scale evaluations (like the Codex paper), it is common to generate a larger number of $n$ total samples (where $n \geq k$) and count the number of correct samples $c$. The unbiased estimator for Pass@k is then calculated as:
$$\text{pass@}k=\underset{\text{samples}}{\mathbb{E}}\left[1-\frac{\binom{n-c}{k}}{\binom{n}{k}}\right]$$
Our implementation simplifies this by strictly generating up to $k$ samples and stopping early upon the first success to save compute.

In [ ]:
# Do not change anything in this cell block

def pass_at_k(model, dataset, k=5, max_new_tokens=256, temperature=0.7, top_p=0.95, do_sample=True, repetition_penalty=1.2):
    """
    Evaluates the Pass@k metric for a given model and dataset.

    For each problem, the model is allowed to generate up to 'k' attempts.
    If any of the 'k' attempts contains the correct answer, the problem is
    marked as successful. Generation stops early if a correct answer is found
    to conserve computational resources.

    Args:
        model (BaseModel): The loaded model wrapper to evaluate.
        dataset (list): A list of dictionaries containing 'question' and 'answer'.
        k (int): The maximum number of generations allowed per problem. Defaults to 5.
        max_new_tokens (int): The maximum length of the generated response. Defaults to 256.
        temperature (float): Sampling temperature. Defaults to 0.7.
        top_p (float): Nucleus sampling probability. Defaults to 0.95.
        do_sample (bool): Whether to use sampling (required for multiple unique attempts). Defaults to True.
        repetition_penalty (float): Penalty for repeating tokens. Defaults to 1.2.

    Returns:
        float: The Pass@k accuracy as a decimal between 0.0 and 1.0.
    """
    correct = 0

    for example in tqdm(dataset, desc=f"Evaluating pass@{k}"):
        question = example["question"]
        gold = example["answer"]

        success = False

        for _ in range(k):
            pred_text = model.generate(
                input_text=question,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=do_sample,
                repetition_penalty=repetition_penalty
            )
            pred = extract_answer(pred_text)

            if pred == gold:
                success = True
                break # Early stopping if we find a correct answer within the 'k' attempts

        if success:
            correct += 1

    return correct / len(dataset)

### Generation Parameters

These parameters will be used for the Pass@1 and Pass@K functions below, when evaluating all the sections (SFT, DPO, and optionally GRPO). These parameters will also be used for model generation examples that we show in the notebook.

**Explanation of Parameters**

- `MAX_NEW_TOKENS`: The maximum number of tokens the model is allowed to generate in the response. This limits the length of the generated output and prevents the model from producing excessively long answers. Note that this is different from the Max Length parameter that we will see in the Training configuration in below sections. Max Length also includes the length of the prompt, while Max New Tokens strictly limits the number of tokens generated.

- `TEMPERATURE`: Controls the randomness of the token sampling. Lower values (e.g., 0.2-0.5) make the output more deterministic and focused, while higher values (e.g., 1.0 more) increase randomness and diversity. A value of 0.7 provides a balance between coherence and creativity.

- `TOP_P`: Also called nucleus sampling. Instead of sampling from all possible tokens, the model selects from the smallest set of tokens whose cumulative probability is at least `p` (here, set to 95%). This helps keep generation diverse while avoiding very unlikely tokens.

- `REPETITION_PENALTY`: Discourages the model from repeating the same tokens or phrases by reducing the probability of tokens that have already appeared in the generated text. Values greater than 1 increase the penalty, helping reduce repetitive outputs.

In [ ]:
# ==== BEGIN EVALUATION PORTION

# BEGIN CODE : gen_params

# Default parameters are given here. You can keep them as is, but you can also experiment
# and see how the model generations and evaluations change.

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.95
REPETITION_PENALTY = 1.2

# END CODE

# ==== END EVALUATION PORTION

## SFT - Supervised Fine-Tuning

Arya runs the base model on a few sample questions and is immediately humbled.
The model produces grammatically plausible but mathematically nonsensical
answers. It has never been explicitly trained on this kind of step-by-step
problem-solving.

The first and most fundamental fix is **Supervised Fine-Tuning (SFT)**: show the
model thousands of (question, reasoning, answer) examples from the training
data, and train it to imitate the provided solutions.

Technically, SFT is standard causal language modelling. The model learns to
predict the next token given all previous tokens. The key design choice is that
we only compute the training loss on the response (reasoning + answer) tokens,
not on the question tokens. We don't want the model to "memorise" the question;
we want it to learn how to respond to it.

### SFT Dataset

In [ ]:

# You can write code in some portions of this cell block, marked with BEGIN CODE and END CODE.

# ==== BEGIN EVALUATION PORTION

class SFTDataset(Dataset):
    """
    PyTorch Dataset for Supervised Fine-Tuning (SFT) on arithmetic reasoning tasks.
    Prepares input text sequences and masks the prompt labels so the loss is only
    computed on the generated reasoning and answer.
    """
    def __init__(self, data: list, tokenizer, max_length: int = 512):
        """
        Args:
            data (list): A list of dictionaries containing 'question', 'reasoning', and 'answer'.
            tokenizer: The Hugging Face tokenizer to convert text to IDs.
            max_length (int): Maximum sequence length for truncation. Defaults to 512.
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        """Returns the total number of items in the dataset."""
        return len(self.data)

    def _format_samples(self, item: dict) -> tuple:
        """
        Formats a single example into a prompt and a response string for Chain of Thought SFT.

        Args:
            item (dict): A dictionary with keys 'question', 'reasoning', 'answer'.

        Returns:
            tuple: (prompt, response) where
                - prompt (str): The input prompt shown to the model (question).
                - response (str): The target completion containing reasoning and answer.

        Example:
            >>> item = {
                'question': 'What is 2% of 200?',
                'reasoning': '2% of 200 is 2/100 * 200 = 4',
                'answer': '4'}

            >>> self._format_samples(item)
                ('What is 2% of 200?\n', '2% of 200 is 2/100 * 200 = 4\n####4<EOS>')

        Notes:
            - Append a newline '\n' to the end of question and reasoning.
            - Put the answer immediately after four hashes, without any spaces, as shown in the example.
            - Add the <EOS> token from the tokenizer, which is the End of Sequence token at the end.
        """
        # BEGIN CODE : sft_data._format_samples
        # ADD YOUR CODE HERE

        # END CODE

    def _apply_causal_mask(self, input_ids: torch.Tensor, prompt_length: int, ignore_index: int = -100) -> torch.Tensor:
        """
        Creates the label tensor for causal language modeling by masking out the prompt tokens.

        Args:
            input_ids (torch.Tensor): The full tokenized sequence (prompt + answer).
            prompt_length (int): The number of tokens that make up the prompt.
            ignore_index (int): The value to use for ignored tokens in the loss calculation.

        Returns:
            torch.Tensor: A 1D tensor of the same length as input_ids, with the prompt masked.

        Example:
            >>> input_ids = torch.Tensor([402, 1054, 2035, 12023, 20983, 50256])
            >>> prompt_length = 2
            >>> ignore_index = -100

            >>> self._apply_causal_mask(input_ids, prompt_length, ignore_index)
                torch.Tensor([-100, -100, 2035, 12023, 20983, 50256])

        Note:
            - The function should NOT change the original input_ids passed to it.
        """
        # BEGIN CODE : sft_data._apply_causal_mask
        # ADD YOUR CODE HERE

        # END CODE

    def __getitem__(self, idx: int) -> dict:
        """
        Retrieves and tokenizes a single data pair, structuring it for causal language modeling.

        Args:
            idx (int): The index of the item to retrieve.

        Returns:
            dict: A dictionary containing three 1D PyTorch tensors:
                - 'input_ids' (torch.Tensor): The tokenized sequence (question + reasoning + answer).
                                              Shape: (sequence_length,)
                - 'attention_mask' (torch.Tensor): 1s for real tokens, 0s for padding.
                                                   Shape: (sequence_length,)
                - 'labels' (torch.Tensor): Identical to input_ids, but tokens corresponding to the
                                           'question' string are replaced with -100 to ignore them
                                           during loss computation. Shape: (sequence_length,)
        """

        item = self.data[idx]

        # Format input (question) and target (reasoning + answer)
        prompt_text, answer_text = self._format_samples(item)

        # Tokenize separately to easily find prompt length
        prompt_tokens = self.tokenizer(prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(0)
        answer_tokens = self.tokenizer(answer_text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(0)

        # Concatenate and enforce max length
        input_tokens = torch.cat([prompt_tokens, answer_tokens], dim=0)
        input_tokens = input_tokens[:self.max_length]

        # Create attention mask
        attention_mask = torch.ones(len(input_tokens), dtype=torch.long)

        # Apply causal mask using the helper function
        prompt_length = len(prompt_tokens)

        # Adjust prompt_length if the sequence was truncated heavily into the prompt itself
        # (Edge case handling to prevent out-of-bounds indexing)
        prompt_length = min(prompt_length, self.max_length)

        labels = self._apply_causal_mask(input_tokens, prompt_length)

        return {
            "input_ids": input_tokens,
            "attention_mask": attention_mask,
            "labels": labels
        }

# Each training example has a different length (some problems are short, some are
# long). But GPU operations work most efficiently on rectangular tensors where
# every sequence in a batch has the same length.

# `sft_collate_fn` is called by PyTorch's DataLoader to combine a list of
# individual examples into a single batch tensor. It finds the longest sequence in
# the batch and pads all shorter sequences with the tokenizer's special pad token
# on the right. Padding positions are masked with 0 in the attention mask (so the
# model ignores them) and with -100 in the labels (so they don't contribute to the loss).
def sft_collate_fn(batch: list, pad_token_id: int) -> dict:
    """
    Collate function to dynamically pad batches of sequences to the length of the longest
    sequence within the specific batch, ensuring consistent tensor shapes for the model.

    Args:
        batch (list): A list of dictionaries, where each dictionary represents a single data point
                      (returned by `SFTDataset.__getitem__`).
        pad_token_id (int): The ID to use for padding the input IDs.

    Returns:
        dict: A dictionary of batched and padded PyTorch tensors.
              The tensors 'input_ids', 'attention_mask', and 'labels' will all
              have the shape (batch_size, max_sequence_length_in_batch).
    """

    # Find the maximum sequence length in the current batch
    max_batch_length = max(item["input_ids"].size(0) for item in batch)

    input_ids_list, attention_mask_list, labels_list = [], [], []

    for item in batch:
        seq_len = item["input_ids"].size(0)
        padding_length = max_batch_length - seq_len

        # Pad tokens to the right (append padding to end of sequence)
        padded_input_ids = F.pad(item["input_ids"], (0, padding_length), value=pad_token_id)
        padded_attention_mask = F.pad(item["attention_mask"], (0, padding_length), value=0)
        padded_labels = F.pad(item["labels"], (0, padding_length), value=-100) # Ensure padding in labels is ignored in loss

        input_ids_list.append(padded_input_ids)
        attention_mask_list.append(padded_attention_mask)
        labels_list.append(padded_labels)

    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list)
    }


# ==== END EVALUATION PORTION

### SFT Trainer

The `SFTTrainer` is the engine that runs the actual training loop. It ties
together the model, the data, the optimiser, and the loss function.

**`_shift_cross_entropy`** implemented below is the mathematical heart of causal language modelling.
A GPT-style model at position `t` produces a probability distribution over the
entire vocabulary, a prediction of what the next token (at position `t+1`) will be.
To compute the loss, we need to align predictions with targets:
  - Logit at position 0 should match label at position 1
  - Logit at position 1 should match label at position 2
and so on.

This "shift by one" is achieved by slicing off the last logit and the first label,
then calling PyTorch's cross-entropy. The `ignore_index=-100` argument ensures that
masked prompt tokens don't contribute to the loss.

In [ ]:
# You can write code in some portions of this cell block, marked with BEGIN CODE and END CODE.

# ==== BEGIN EVALUATION PORTION

class SFTTrainer:
    """
    Custom PyTorch trainer for handling Supervised Fine-Tuning loops.
    """
    def __init__(self, base_model, train_loader, eval_loader, num_epochs: int, optimizer, scheduler, device: str = None, ignore_index: int = -100):
        """
        Args:
            base_model (BaseModel): The wrapper model object.
            train_loader (DataLoader): PyTorch DataLoader for the training dataset.
            eval_loader (DataLoader): PyTorch DataLoader for the evaluation dataset.
            num_epochs (int): Number of training epochs.
            optimizer: PyTorch optimizer (e.g., AdamW).
            scheduler: PyTorch learning rate scheduler.
            device (str): Device to train on ('cpu' or 'cuda').
        """
        self.base_model = base_model
        self.model = base_model.model.to(device)
        self.tokenizer = base_model.tokenizer
        self.train_loader = train_loader
        self.eval_loader = eval_loader
        self.num_epochs = num_epochs
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.save_path = "sft_model"
        self.ignore_index = ignore_index
        self.device = device

        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.train_losses = []
        self.eval_losses = []

    def _shift_cross_entropy(self, logits: torch.Tensor, labels: torch.Tensor, ignore_index: int = -100) -> torch.Tensor:
        """
        Aligns the logits and labels for autoregressive next-token prediction and computes the cross-entropy loss.

        In causal language modeling, the model predicts the token at position `t+1` given the tokens up to position `t`.
        To compute the loss correctly, this function shifts the `logits` tensor to exclude the last timestep,
        and shifts the `labels` tensor to exclude the first timestep, ensuring predictions align with their targets.

        Args:
            logits (torch.Tensor): The raw, unnormalized scores from the model's language modeling head.
                                   Expected shape: (batch_size, sequence_length, vocab_size).
            labels (torch.Tensor): The ground truth token IDs.
                                   Expected shape: (batch_size, sequence_length).
            ignore_index (int, optional): The target label value that is ignored and does not contribute
                                          to the input gradient (usually applied to prompt or padding tokens).
                                          Defaults to -100.

        Returns:
            torch.Tensor: A scalar tensor containing the computed cross-entropy loss.

        Example:
            >>> # Batch size 1, Sequence Length 3, Vocab Size 2

            >>> logits = torch.tensor([[[2.5, -1.0],   # Step 0 (predicts Step 1)
            ...                         [0.5,  1.5],   # Step 1 (predicts Step 2)
            ...                         [1.0,  1.0]]]) # Step 2 (predicts Step 3)

            >>> labels = torch.tensor([[-100, 1, 0]])

            >>> ignore_index = -100

            >>> loss = self._shift_cross_entropy(logits, labels, ignore_index)

            >>> print(loss)
                torch.tensor(2.4215)

        Note:
            - The tensors must be contiguous in memory before reshaping for the cross-entropy function.

        Hint: Check documentation for torch's cross entropy function.
        """
        # BEGIN CODE : sft_trainer._shift_cross_entropy
        # ADD YOUR CODE HERE


        # END CODE


    def train_epoch(self) -> float:
        """
        Executes a single epoch of training over the train_loader.

        Returns:
            float: The average training loss for the epoch.
        """
        self.model.train()
        total_loss = 0
        for batch in tqdm(self.train_loader, desc="Training"):
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels = batch['labels'].to(self.device)

            # Forward pass
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

            loss = self._shift_cross_entropy(outputs.logits, labels, self.ignore_index)

            # Step Optimizer
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(self.train_loader)
        self.train_losses.append(avg_loss)
        return avg_loss

    def eval_epoch(self) -> float:
        """
        Executes a single epoch of evaluation over the eval_loader.

        Returns:
            float: The average evaluation loss for the epoch.
        """
        self.model.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in tqdm(self.eval_loader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                # Forward pass
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

                loss = self._shift_cross_entropy(outputs.logits, labels, self.ignore_index)

                total_loss += loss.item()

        avg_loss = total_loss / len(self.eval_loader)
        self.eval_losses.append(avg_loss)
        return avg_loss

    def train(self):
        """
        Executes the full training loop across all specified epochs and saves the model.
        """
        num_epochs = self.num_epochs
        for epoch in range(num_epochs):
            train_loss = self.train_epoch()
            eval_loss = self.eval_epoch()

            print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f} - Eval Loss: {eval_loss:.4f}")
        self.base_model.save(self.save_path)

# ==== END EVALUATION PORTION

### Parameters and Config for Training

In [ ]:
# ==== BEGIN EVALUATION PORTION

# BEGIN CODE : sft_params
# ADD YOUR CODE HERE
# Default Parameters are given here. You can keep them as is, but you can also experiment if you wish to.

MAX_LENGTH = 512
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3

WEIGHT_DECAY = 0.01

# END CODE

# ==== END EVALUATION PORTION

base_model = BaseModel()

COLLATE_FN = lambda batch: sft_collate_fn(batch, base_model.tokenizer.pad_token_id)

train_dataset = SFTDataset(train_data, base_model.tokenizer, max_length = MAX_LENGTH)
eval_dataset = SFTDataset(eval_data, base_model.tokenizer, max_length = MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=False, collate_fn=COLLATE_FN)

eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, drop_last=False, collate_fn=COLLATE_FN)

optimizer = torch.optim.AdamW(base_model.model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * len(train_loader))

In [ ]:
# Do not change anything in this cell block

# Training may take on the order of ~10 mins per epoch on Colab.
sft_trainer = SFTTrainer(base_model, train_loader, eval_loader, NUM_EPOCHS, optimizer, scheduler)
sft_trainer.train()

In [ ]:
# Clear memory
flush_memory(base_model, sft_trainer, optimizer, scheduler)

del base_model, sft_trainer, optimizer, scheduler

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Evaluation

Let's look at sample generations for both the Base Model and the Fine-Tuned Model. You may see the base model generating until the max length limit, as it is only trained for text generation, not for instruction following and knowing when to stop.

The fine-tuned model, on the other hand, shows some chain-of-thought, and the responses may look shorter and more relevant.

In [ ]:
# Load Base Model and Fine Tuned Model
base_model = BaseModel()

sft_model = BaseModel(r'./sft_model')

In [ ]:
print("Sample Base Model Generations\n")

for data in sample_data:
    pred = base_model.generate(
        data['question'],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY
    )
    print(f"Question:\n {data['question']}\n")
    print(f"Output:\n {pred}")
    print("---"*20)
    print("\n")

In [ ]:
print("Sample Fine Tuned Model Generations\n")

for data in sample_data:
    pred = sft_model.generate(
        data['question'],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY
    )
    print(f"Question:\n {data['question']}\n")
    print(f"Output:\n {pred}")
    print("---"*20)
    print("\n")

In [ ]:
# Pass@1
# You may see scores in the range of 0.01 - 0.05, maybe more if the fine-tuning went really well.

print("Evaluating Base Model Pass@1")
base_pass1 = pass_at_1(
    base_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print("\nEvaluating SFT Model Pass@1")
sft_pass1 = pass_at_1(
    sft_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print("\n\n")
print(f"Base Model Pass@1: {base_pass1}")
print(f"SFT Model Pass@1: {sft_pass1}")


In [ ]:
# Pass@K
# The Pass@K performance should be noticeably better.
# You may now also notice that the evaluation of Base Model takes much longer
# This is because the Base Model doesn't really know when to stop and output EOS.
# The Base Model is just a text-generation model.
# On the other hand, the SFT Model has learnt to give better responses and stop,
# leading to faster evaluation.

print("Evaluating Base Model Pass@K")
base_passK = pass_at_k(
    base_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print("\nEvaluating SFT Model Pass@K")
sft_passK = pass_at_k(
    sft_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print("\n\n")
print(f"Base Model Pass@K: {base_passK}")
print(f"SFT Model Pass@K: {sft_passK}")

In [ ]:
# Clear memory
flush_memory(base_model, sft_model)

del base_model, sft_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## DPO - Direct Preference Optimization

The SFT model is better, but Arya notices something frustrating: the model
sometimes produces plausible-sounding but wrong solutions. It will confidently
walk through a series of steps, but make an arithmetic error halfway through
and arrive at a wrong answer without any sign of doubt.

The root cause is that SFT only teaches the model to imitate training examples.
It has no mechanism for the model to learn that some answers are better than
others. If a training example happens to have a slightly convoluted solution,
the model might imitate that too.

What Arya needs is a way to tell the model: "When you see this problem, the
reasoning in example A is preferable to the reasoning in example B." This is
the job of **preference learning**.

The classic approach is RLHF (Reinforcement Learning from Human Feedback), which
trains a separate reward model and then optimises the policy against it using PPO.
But that is complex, compute-heavy, and can be unstable.

**DPO (Direct Preference Optimisation)**, Rafailov et al. 2023, is an elegant
shortcut. It shows that the optimal RLHF policy has a closed form. You can
directly optimise the language model to prefer chosen responses over rejected ones
without training a separate reward model. The DPO loss is simply:

$$
    \mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x, y_w, y_l)}
    \left[ \log \sigma \left(
        \beta \left(
            \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}
            - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}
        \right)
    \right) \right]
$$

where:
  - $y_w$ is the **chosen** (preferred/correct) completion
  - $y_l$ is the **rejected** (dispreferred/incorrect) completion
  - $\pi_\theta$ is the model being trained
  - $\pi_{\text{ref}}$ is a frozen copy of the SFT model (the reference)
  - $\beta$ controls how far the trained model is allowed to deviate from the reference

Intuitively, we increase the probability of the correct answer and decrease
the probability of the wrong answer, but don't stray too far from the SFT model
(the $\beta$ term acts as a leash).

For every math question, the DPO dataset provides:
  1. **chosen**: the correct, well-reasoned solution
  2. **rejected**: an incorrect or poorly-reasoned solution


### DPO Dataset

Each item in the DPO dataset looks like:
```json
{
  "question":  "If Riya has 12 stickers...",
  "chosen":    "Step 1: ... Step 2: ... #### 7",
  "rejected":  "We add all numbers: 12 + 3 = 15. #### 15"
}
```

In [ ]:
# Do not change anything in this cell block

# Preference Dataset based on GSM8K

dpo_train_file = "./data/dpo_train.jsonl"
dpo_eval_file = "./data/dpo_eval.jsonl"

dpo_train_data = load_jsonl(dpo_train_file)
dpo_eval_data = load_jsonl(dpo_eval_file)

In [ ]:
# You can write code in some portions of this cell block, marked with BEGIN CODE and END CODE.

# ==== BEGIN EVALUATION PORTION

class DPODataset(Dataset):
    """
    Dataset for Direct Preference Optimization (DPO).
    Processes prompts (i.e. questions) alongside their chosen and rejected completions.
    """
    def __init__(self, data, tokenizer, max_length: int = 512):
        """
        Args:
            data (list): List of dictionaries with 'question', 'chosen', and 'rejected' keys.
            tokenizer: Tokenizer from BaseModel().
            max_length (int): Maximum token limit for the combined sequence.
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        "Returns number of data samples"
        return len(self.data)

    def _tokenize_pair(self, question: str, response: str):
        """
        Concatenates and tokenizes a question-response pair while masking the question (i.e. prompt) for loss calculation.

        This method prepares the data for Causal Language Modeling by joining the instruction and
        the completion. It specifically handles 'loss masking' by setting the label values for
        the prompt tokens to -100, ensuring the model is only optimized to predict the response
        tokens and not the input prompt.

        Args:
            question (str): The input prompt or instruction provided to the model.
            response (str): The specific completion (chosen or rejected) to be tokenized.

        Returns:
            tuple: A tuple containing three 1D Torch tensors, all having the same size:
                - input_ids (torch.LongTensor): The concatenated token IDs of the prompt and
                  response, truncated to `self.max_length`.
                - attention_mask (torch.LongTensor): A bitmask indicating non-padding tokens
                  (1 for all tokens in this stage).
                - labels (torch.LongTensor): Target IDs where prompt tokens are masked with -100
                  and response tokens are kept for loss computation.

        Example:
            >>> # question="Hi, send help", response="No", max_length=10
            >>> # prompt_tokens = [10, 15, 20] ("Hi, send help\n")
            >>> # answer_tokens = [50, 2]     ("No", "<EOS>")

            >>> input_ids, mask, labels = dataset._tokenize_pair("Hi, send help", "No")

            >>> print(input_ids)
                torch.tensor([10, 15, 20, 50, 2])

            >>> print(mask)
                torch.tensor([1, 1, 1, 1, 1])

            >>> print(labels)
                torch.tensor([-100, -100, -100, 50, 2])

        Note:
            - At this stage the attention masks should be all 1's as we will have to do Dynamic Padding
              so the corresponding masks are handled in the dpo_collate_fn.
            - Append the response with the tokenizer's EOS token.
            - In masking, use -100, which is the default `ignore_index` for PyTorch's CrossEntropyLoss.
        """
        # BEGIN CODE : dpo_data._tokenize_pair
        # ADD YOUR CODE HERE

        # END CODE

    def __getitem__(self, idx: int):
        """
        Retrieves fully tokenized and masked tensors for both chosen and rejected completions.

        This method accesses a data sample by index and processes the prompt alongside
        both the preferred ('chosen') and non-preferred ('rejected') responses to
        create the paired inputs required for the DPO loss objective.

        Args:
            idx (int): Index of the dataset sample to retrieve.

        Returns:
            dict: A dictionary containing six PyTorch tensors:
                - chosen_input_ids: Tokens for prompt + chosen response.
                - chosen_attention_mask: Attention mask for the chosen pair.
                - chosen_labels: Masked labels for the chosen pair.
                - rejected_input_ids: Tokens for prompt + rejected response.
                - rejected_attention_mask: Attention mask for the rejected pair.
                - rejected_labels: Masked labels for the rejected pair.

        Example:
            >>> # Data: {"question": "Hi, send help", "chosen": "Sure!", "rejected": "No"}
            >>> batch = dataset[0]
            >>> print(batch["chosen_input_ids"].shape)
                torch.Size([7])
            >>> print(batch["rejected_input_ids"].shape)
                torch.Size([5])

        Note:
            - Both pairs use the same prompt tokens but different response tokens.
            - Tensors are returned as 1D and are typically padded later in the collate function.
        """
        item = self.data[idx]
        question = item["question"]

        chosen_ids, chosen_mask, chosen_labels = self._tokenize_pair(
            question, item["chosen"]
        )

        rejected_ids, rejected_mask, rejected_labels = self._tokenize_pair(
            question, item["rejected"]
        )

        return {
            "chosen_input_ids": chosen_ids,
            "chosen_attention_mask": chosen_mask,
            "chosen_labels": chosen_labels,

            "rejected_input_ids": rejected_ids,
            "rejected_attention_mask": rejected_mask,
            "rejected_labels": rejected_labels,
        }

def dpo_collate_fn(batch, pad_token_id: int):
    """
    Dynamically pads a batch of DPO samples to the maximum sequence length within the batch.

    DPO requires comparing log-probabilities of 'chosen' and 'rejected' pairs. To
    process these in parallel, this function identifies the longest sequence across
    both types in the current batch and pads all others to match that length.

    Args:
        batch (list): A list of dictionaries from DPODataset.__getitem__.
        pad_token_id (int): The ID used for padding tokens (from the tokenizer).

    Returns:
        dict: A dictionary of batched and padded tensors.
              Each tensor has shape (batch_size, max_sequence_length).

    Example:
        >>> # Batch with two samples of lengths [5, 7] and [4, 6]
        >>> # max_len in batch = 7

        >>> padded_batch = dpo_collate_fn(batch, pad_token_id=0)

        >>> print(padded_batch["chosen_input_ids"].shape)
            torch.Size([2, 7])

        >>> print(padded_batch["chosen_labels"][0])
            torch.tensor([-100, -100, 50, 2, -100, -100, -100]) # Example padding

    Note:
        - Labels are padded with -100 so that padding tokens are ignored in loss.
        - Attention masks are padded with 0 to ensure the model ignores padding tokens.
        - Right-padding is used to maintain consistent token alignment for causal models.
    """
    max_len = max(
        max(item["chosen_input_ids"].size(0), item["rejected_input_ids"].size(0))
        for item in batch
    )

    chosen_ids, chosen_masks, chosen_labels = [], [], []
    rejected_ids, rejected_masks, rejected_labels = [], [], []

    for item in batch:
        # Calculate padding needed for current item
        c_pad = max_len - item["chosen_input_ids"].size(0)
        r_pad = max_len - item["rejected_input_ids"].size(0)


        # Right-pad chosen sequences
        chosen_ids.append(
            F.pad(item["chosen_input_ids"], (0, c_pad), value=pad_token_id)
        )
        chosen_masks.append(
            F.pad(item["chosen_attention_mask"], (0, c_pad), value=0)
        )
        chosen_labels.append(
            F.pad(item["chosen_labels"], (0, c_pad), value=-100)
        )

        # Right-pad rejected sequences
        rejected_ids.append(
            F.pad(item["rejected_input_ids"], (0, r_pad), value=pad_token_id)
        )
        rejected_masks.append(
            F.pad(item["rejected_attention_mask"], (0, r_pad), value=0)
        )
        rejected_labels.append(
            F.pad(item["rejected_labels"], (0, r_pad), value=-100)
        )

    return {
        "chosen_input_ids": torch.stack(chosen_ids),
        "chosen_attention_mask": torch.stack(chosen_masks),
        "chosen_labels": torch.stack(chosen_labels),

        "rejected_input_ids": torch.stack(rejected_ids),
        "rejected_attention_mask": torch.stack(rejected_masks),
        "rejected_labels": torch.stack(rejected_labels),
    }

# ==== END EVALUATION PORTION

### DPO Trainer

Direct Preference Optimization (DPO) reformulates the RLHF objective to bypass the need for a
separate reward model. It uses the language model itself as the reward model.

#### Class Notation

In class, DPO was written as an argmax objective over the LLM parameters θ:

$$
\underset{\theta}{\text{argmax}} \sum_{i=1}^{n} \log \sigma \left( \beta \log \frac{P_\theta(y_i^w \mid x_i)}{P_{\theta_0}(y_i^w \mid x_i)} - \beta \log \frac{P_\theta(y_i^l \mid x_i)}{P_{\theta_0}(y_i^l \mid x_i)} \right)
$$

Where:
* $P_\theta$ is the **policy model** being actively trained (parameterised by $\theta$).
* $P_{\theta_0}$ is the frozen **reference model** (parameterised by the original weights $\theta_0$).
* $y_i^w$ is the **w**on (chosen) response, $y_i^l$ is the **l**ost (rejected) response.

#### Standard Notation

The same objective is written in the literature as a minimisation loss:

$$
\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\mathbb{E}_{(x, y_w, y_l) \sim D} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w | x)}{\pi_{ref}(y_w | x)} - \beta \log \frac{\pi_\theta(y_l | x)}{\pi_{ref}(y_l | x)} \right) \right]
$$

Where:
* $\pi_\theta$ is the **policy model** being actively trained.
* $\pi_\text{ref}$ is the frozen **reference model** (usually the SFT model).
* $x$ is the prompt, $y_w$ is the chosen response, and $y_l$ is the rejected response.
* $\beta$ controls the strength of the KL divergence penalty (how close $\pi_\theta$ should stay
  to $\pi_\text{ref}$).
* $\sigma$ is the sigmoid function.


The two formulations are identical in meaning, just flipped in sign and swapped in notation:

| Class | This Assignment |
|---|---|
| $P_\theta$ | $\pi_\theta$ |
| $P_{\theta_0}$ | $\pi_\text{ref}$ |
| $y^w$ (won) | $y_w$ (chosen) |
| $y^l$ (lost) | $y_l$ (rejected) |
| **argmax** $\sum$ (maximise sum) | **min** $-\mathbb{E}[\cdot]$ (minimise negative expectation) |

Maximising $\log \sigma(\cdot)$ is exactly the same as minimising $-\log \sigma(\cdot)$,
so the gradient signal is identical.

The class formula uses a **sum** over $n$ examples, while in this code we expect you to use mean in `_compute_preference_loss()` below.

Using the mean instead of the sum keeps the loss magnitude stable regardless of batch size, which makes learning rate tuning easier. It does not change which $\theta$ is optimal.

In [ ]:
# You can write code in some portions of this cell block, marked with BEGIN CODE and END CODE.

# ==== BEGIN EVALUATION PORTION

class DPOTrainer:
    """
    Trainer class for Direct Preference Optimization.
    Manages the active policy model and the frozen reference model to compute the DPO loss.
    """
    def __init__(self, base_model, ref_base_model, train_loader, eval_loader,
                 num_epochs, optimizer, scheduler, device: str = None, beta: float = 0.1):
        """
        Args:
            base_model: The active policy model wrapper (will be updated).
            ref_base_model: The frozen reference model wrapper (used for baseline log-probs).
            train_loader: DataLoader for training preference data.
            eval_loader: DataLoader for evaluation preference data.
            num_epochs: Total number of training epochs.
            optimizer: PyTorch optimizer.
            scheduler: PyTorch learning rate scheduler.
            device: Computing device ('cuda' or 'cpu').
            beta (float): Temperature parameter controlling deviation from the reference model.
        """
        self.base_model = base_model
        self.model = base_model.model.to(device)

        # Freeze the reference model completely
        self.ref_model = ref_base_model.model.to(device)
        self.ref_model.eval()
        for p in self.ref_model.parameters():
            p.requires_grad = False

        self.tokenizer = base_model.tokenizer
        self.train_loader = train_loader
        self.eval_loader = eval_loader
        self.num_epochs = num_epochs
        self.optimizer = optimizer
        self.scheduler = scheduler

        self.device = device
        if not device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.beta = beta
        self.save_path = "dpo_model"

        self.train_losses = []
        self.eval_losses = []

    def _get_log_probs(self, logits: torch.tensor, labels: torch.tensor):
        """
        Extracts and averages the log probabilities of the actual target tokens.

        This function aligns logits with target labels using a standard causal shift,
        extracts the log-likelihood for each gold token, and computes the average
        log probability per sequence. It ensures that masked tokens (labeled -100)
        do not contribute to the final sum or the averaging denominator.

        Args:
            logits (torch.Tensor): Raw, unnormalized scores from the model.
                                   Expected shape: (batch_size, sequence_length, vocab_size).
            labels (torch.Tensor): Ground truth token IDs with -100 for ignored/prompt tokens.
                                   Expected shape: (batch_size, sequence_length).

        Returns:
            torch.Tensor: A 1D tensor of shape (batch_size,) where each element is the
                          mean log probability of the non-masked tokens in that sequence.

        Example:
            >>> # Batch size 2, Sequence length 3, Vocab size 3

            >>> logits = torch.tensor([[[-0.1570, -0.3147, -2.4980],
                                        [-1.1140,  0.8445, -0.0966],
                                        [ 0.8301, -0.5009, -1.3697]],

                                       [[ 0.1553, -0.2213, -0.2012],
                                        [-0.4858,  0.9677, -0.5543],
                                        [ 0.7617,  0.8692, -0.4151]]])

            >>> labels = torch.tensor([[-100,    1,    2],
                                       [-100,    0,    1]])

            >>> logps = self._get_log_probs(logits, labels)

            >>> print(logps)
                tensor([-1.0964, -0.6214])

        Note:
            - Logits at index `t` are used to predict the label at index `t+1`.
            - We use `torch.gather` to pull specific vocab indices from the log-softmax output.
            - Valid tokens are identified where `labels != -100`.
        """
        # BEGIN CODE : dpo_trainer._get_log_probs
        # ADD YOUR CODE HERE

        # END CODE

    def _compute_implicit_rewards(self, policy_chosen_log_probs: torch.Tensor, policy_rejected_log_probs: torch.Tensor,
        ref_chosen_log_probs: torch.Tensor, ref_rejected_log_probs: torch.Tensor) -> tuple:
        """
        Calculates the implicit rewards for chosen and rejected completions based on the DPO framework.

        In DPO, the reward is implicitly defined as the scaled log-ratio of the current policy's
        probabilities relative to the reference model's probabilities. This function computes
        the difference in log-probabilities, which represents how much more (or less) likely
        the current policy finds a completion compared to the frozen reference model.

        Args:
            policy_chosen_log_probs (torch.Tensor): Log probabilities of chosen responses from
                                                    the model being trained. Shape: (batch_size,).
            policy_rejected_log_probs (torch.Tensor): Log probabilities of rejected responses from
                                                      the model being trained. Shape: (batch_size,).
            ref_chosen_log_probs (torch.Tensor): Log probabilities of chosen responses from
                                                 the frozen reference model. Shape: (batch_size,).
            ref_rejected_log_probs (torch.Tensor): Log probabilities of rejected responses from
                                                   the frozen reference model. Shape: (batch_size,).

        Returns:
            tuple: A tuple containing two 1D tensors:
                - chosen_rewards (torch.Tensor): Implicit reward for preferred completions.
                - rejected_rewards (torch.Tensor): Implicit reward for non-preferred completions.

        Example:
            >>> # Batch size of 2
            >>> policy_chosen_log_probs = torch.tensor([-0.5, -1.1])
            >>> policy_rejected_log_probs = torch.tensor([-2.0, -1.5])
            >>> ref_chosen_log_probs = torch.tensor([-1.2, -0.9])
            >>> ref_rejected_log_probs = torch.tensor([-1.8, -1.8])

            >>> chosen_rewards, rejected_rewards = self._compute_implicit_rewards(
            ...     policy_chosen_log_probs, policy_rejected_log_probs,
            ...     ref_chosen_log_probs, ref_rejected_log_probs
            ... )

            >>> print(chosen_rewards)
            tensor([ 0.7000, -0.2000])
            >>> print(rejected_rewards)
            tensor([-0.2000,  0.3000])

        Note:
            - These rewards are typically multiplied by a hyperparameter (beta) in the final loss.
            - This formulation bypasses the need for an explicit Reward Model (RM).
        """
        # BEGIN CODE : dpo_trainer._compute_implicit_rewards
        # ADD YOUR CODE HERE

        # END CODE

    def _compute_preference_loss(self, chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor, beta: float) -> torch.Tensor:
        """
        Computes the DPO preference loss using the Bradley-Terry model of preferences.

        This function calculates the final training objective by taking the difference
        between the implicit rewards of the chosen and rejected completions. The difference
        is scaled by the hyperparameter `beta` and passed through a log-sigmoid function
        to maximize the likelihood that the chosen response is preferred over the rejected one.

        Args:
            chosen_rewards (torch.Tensor): Implicit rewards for the preferred responses.
                                           Shape: (batch_size,).
            rejected_rewards (torch.Tensor): Implicit rewards for the non-preferred responses.
                                             Shape: (batch_size,).
            beta (float): Temperature parameter that controls the strength of the KL penalty
                          relative to the reference model. Typical values range from 0.1 to 0.5.

        Returns:
            torch.Tensor: A scalar tensor representing the mean negative log-likelihood
                          loss across the batch.

        Example:
            >>> # Batch size of 2
            >>> chosen_rewards = torch.tensor([0.5, 1.2])
            >>> rejected_rewards = torch.tensor([-0.2, 0.4])
            >>> beta = 0.1

            >>> loss = self._compute_preference_loss(chosen_rewards, rejected_rewards, beta)
            >>> print(loss)
            tensor(0.6564)

        Note:
            - As the difference (chosen - rejected) increases, the loss decreases.
            - This formulation ensures the policy stays close to the reference model
              while satisfying the preference constraints.
        """
        # BEGIN CODE : dpo_trainer._compute_preference_loss
        # ADD YOUR CODE HERE

        # END CODE

    def _compute_loss(self, batch: dict) -> torch.Tensor:
        """
        Orchestrates the full DPO loss computation by comparing the active policy against a reference model.

        This method implements the core DPO algorithm: it performs forward passes for both the
        current policy and the frozen reference model, extracts log probabilities for the
        preferred and rejected completions, and optimizes the policy to maximize the
        likelihood of the preferred response while staying anchored to the reference model.

        Args:
            batch (dict): A dictionary of tensors containing:
                - "chosen_input_ids" & "rejected_input_ids": Token sequences for both pairs.
                - "chosen_attention_mask" & "rejected_attention_mask": Respective attention masks.
                - "chosen_labels" & "rejected_labels": Respective target labels (masked).

        Returns:
            torch.Tensor: A scalar loss value representing the DPO objective for the current batch.

        Example:
            >>> # Mock batch with a batch size of 2 and sequence length of 4
            >>> batch = {
            ...     "chosen_input_ids": torch.tensor([[101, 2054, 2003, 102], [101, 1045, 2066, 102]]),
            ...     "rejected_input_ids": torch.tensor([[101, 2054, 2009, 102], [101, 1045, 2099, 102]]),
            ...     "chosen_attention_mask": torch.tensor([[1, 1, 1, 1], [1, 1, 1, 1]]),
            ...     "rejected_attention_mask": torch.tensor([[1, 1, 1, 1], [1, 1, 1, 1]]),
            ...     "chosen_labels": torch.tensor([[-100, 2054, 2003, 102], [-100, 1045, 2066, 102]]),
            ...     "rejected_labels": torch.tensor([[-100, 2054, 2009, 102], [-100, 1045, 2099, 102]])
            ... }

            >>> loss = self._compute_loss(batch)
            >>> print(loss)
            tensor(0.6931, grad_fn=<NegBackward0>)


        Note:
            - Efficiency: Chosen and rejected sequences are concatenated to use a single
              forward pass per model.
            - Gradient Isolation: The reference model is kept in `eval()` mode and wrapped
              in `torch.no_grad()` to prevent unnecessary memory usage and updates.
            - Device Management: All batch tensors are moved to the designated compute device
              (e.g., CUDA) before processing.
        """
        # BEGIN CODE : dpo_trainer._compute_loss
        # ADD YOUR CODE HERE

        # END CODE

    def train_epoch(self):
        """Runs a single epoch of DPO training."""
        self.model.train()
        total_loss = 0

        for batch in tqdm(self.train_loader, desc="DPO Training"):
            loss = self._compute_loss(batch)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()
            self.train_losses.append(loss.item())

        avg_loss = total_loss / len(self.train_loader)
        return avg_loss

    def eval_epoch(self):
        """Runs a single epoch of evaluation without backpropagation."""
        self.model.eval()
        total_loss = 0

        with torch.no_grad():
            for batch in tqdm(self.eval_loader, desc="DPO Evaluating"):
                loss = self._compute_loss(batch)
                total_loss += loss.item()

        avg_loss = total_loss / len(self.eval_loader)
        self.eval_losses.append(avg_loss)

        return avg_loss

    def train(self):
        """Executes the complete training loop across all epochs and saves the model."""
        for epoch in range(self.num_epochs):
            train_loss = self.train_epoch()
            eval_loss = self.eval_epoch()

            print(
                f"Epoch {epoch+1}/{self.num_epochs} "
                f"- Train Loss: {train_loss:.4f} "
                f"- Eval Loss: {eval_loss:.4f}"
            )

        self.base_model.save(self.save_path)

# ==== END EVALUATION PORTION

### Parameters and Config for Training

DPO uses a lower learning rate compared to SFT. This is
intentional, as the SFT model already knows how to produce math solutions;
DPO just nudges it toward preferring correct ones. Aggressive updates
here would destroy the SFT-learned knowledge.

The $\beta$ hyperparameter controls the strength of the KL constraint.
A higher $\beta$ makes the model stay closer to the SFT reference, while a lower $\beta$
gives it more freedom to deviate in the direction of the preference signal.

In [ ]:
# ==== BEGIN EVALUATION PORTION

# BEGIN CODE : dpo_params
# ADD YOUR CODE HERE
# Default Parameters are given here. You can keep them as is, but you can also experiment if you wish to.
DPO_MAX_LENGTH = 512
DPO_BATCH_SIZE = 4
DPO_LEARNING_RATE = 2e-7
DPO_WEIGHT_DECAY = 0.01
DPO_NUM_EPOCHS = 1
DPO_BETA = 0.4

# END CODE

# ==== END EVALUATION PORTION

# Initialize policy model from SFT weights
dpo_policy_model = BaseModel("sft_model")

# Initialize reference model from SFT weights (this will remain frozen)
dpo_ref_model = BaseModel("sft_model")

collate = lambda batch: dpo_collate_fn(
    batch,
    dpo_policy_model.tokenizer.pad_token_id
)

dpo_train_dataset = DPODataset(
    dpo_train_data,
    dpo_policy_model.tokenizer,
    max_length=DPO_MAX_LENGTH
)

dpo_eval_dataset = DPODataset(
    dpo_eval_data,
    dpo_policy_model.tokenizer,
    max_length=DPO_MAX_LENGTH
)

dpo_train_loader = DataLoader(
    dpo_train_dataset,
    batch_size=DPO_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate
)

dpo_eval_loader = DataLoader(
    dpo_eval_dataset,
    batch_size=DPO_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate
)

dpo_optimizer = torch.optim.AdamW(
    dpo_policy_model.model.parameters(),
    lr=DPO_LEARNING_RATE,
    weight_decay=DPO_WEIGHT_DECAY
)

dpo_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    dpo_optimizer,
    T_max=DPO_NUM_EPOCHS * len(dpo_train_loader)
)

In [ ]:
dpo_trainer = DPOTrainer(
    base_model=dpo_policy_model,
    ref_base_model=dpo_ref_model,
    train_loader=dpo_train_loader,
    eval_loader=dpo_eval_loader,
    num_epochs=DPO_NUM_EPOCHS,
    optimizer=dpo_optimizer,
    scheduler=dpo_scheduler,
    beta=DPO_BETA,
)

dpo_trainer.train()

### Evaluation

It might even so happen that you actually see a degradation in performance after DPO. There could be multiple possible reasons for that.

- The model we are using is very small (~82M). The models that are typically known to perform well after DPO are usually in the billion parameter range.
- Intuitively, DPO does not teach the model new facts or math logic. In some sense,it aligns the model to prefer certain output styles. A tiny model can be intuitively said to have very little internal "knowledge buffer". When you force it to align with preference pairs, it can lose the fragile mathematical patterns it barely learnt during SFT.
- The dataset doesn't have a strong signal. It's a synthetic preference dataset. DPO may result in the model just learning arbitrary stylistic patterns rather than proper reasoning.

In [ ]:
# Load DPO Model

dpo_model = BaseModel('dpo_model')

In [ ]:
# Sample DPO Model Generations

print("Sample DPO Model Generations\n")

for data in sample_data:
    pred = dpo_model.generate(
        data['question'],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY
    )
    print(f"Question:\n {data['question']}\n")
    print(f"Output:\n {pred}")
    print("---"*20)
    print("\n")

In [ ]:
# Pass@1

dpo_pass1 = pass_at_1(
    dpo_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print("\n\n")
print(f"Base Model Pass@1: {base_pass1}")
print(f"SFT Model Pass@1: {sft_pass1}")
print(f"DPO Model Pass@1: {dpo_pass1}")


In [ ]:
# Pass@K

dpo_passK = pass_at_k(
    dpo_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print(f"\nBase Model Pass@K: {base_passK}")
print(f"SFT Model Pass@K: {sft_passK}")
print(f"DPO Model Pass@K: {dpo_passK}")

In [ ]:
# Clear memory
flush_memory(dpo_policy_model, dpo_ref_model, dpo_trainer, dpo_model, dpo_optimizer, dpo_scheduler)

del dpo_policy_model, dpo_ref_model, dpo_trainer, dpo_model, dpo_optimizer, dpo_scheduler

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## GRPO - Group Relative Policy Optimization

**NOTE: GRPO is an Optional Section. This section does not carry any marks, and is here only as an extra, for interested students.**

GRPO is an online reinforcement learning algorithm that simplifies standard Proximal Policy Optimization (PPO). Standard PPO requires a separate "Critic" or "Value" model to estimate the baseline advantage of a state. GRPO eliminates the need for this separate model.

Instead of a learned baseline, GRPO samples a **group** of $G$ different responses for the exact same prompt. The baseline is simply the statistical mean and standard deviation of the rewards within that group.

**1. Group-Relative Advantage:**
For a prompt $x$, we generate $G$ responses $\{y_1, y_2, \dots, y_G\}$. We calculate their rewards $\{r_1, r_2, \dots, r_G\}$. The advantage for the $i$-th response is computed by standardizing the rewards within the group:

$$
    A_i = \frac{r_i - \mu_r}{\sigma_r}
$$
where $\mu_r$ and $\sigma_r$ are the mean and standard deviation of the $G$ rewards.

**2. Importance Sampling Ratio:**
To safely update the policy, we calculate how much more (or less) likely the new policy $\pi_\theta$ is to generate the response compared to the old reference policy $\pi_{ref}$:
$$
    \rho_i(\theta) = \frac{\pi_\theta(y_i|x)}{\pi_{ref}(y_i|x)} = \exp(\log \pi_\theta(y_i|x) - \log \pi_{ref}(y_i|x))
$$

**3. Clipped Surrogate Objective with KL Penalty:**
We want to maximize the reward, but we restrict the policy update from moving too far from the reference model using a clipping mechanism and a Kullback-Leibler (KL) divergence penalty.
$$
    L^{CLIP}(\theta) = -\frac{1}{G} \sum_{i=1}^{G} \min \left( \rho_i(\theta) A_i, \text{clip}(\rho_i(\theta), 1-\epsilon, 1+\epsilon) A_i \right)
$$

$$
    L^{KL}(\theta) = \beta D_{KL}(\pi_\theta || \pi_{ref})
$$

$$
    \mathcal{L}_{Total} = L^{CLIP} + L^{KL}
$$

![GRPO Visual](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/grpo_visual.png)

Image Source: [GRPO Trainer Docs for TRL](https://huggingface.co/docs/trl/grpo_trainer)

### GRPO Dataset

In [ ]:
# Dataset based on GSM8K

grpo_train_file = "./data/grpo_train.jsonl"
grpo_eval_file = "./data/grpo_eval.jsonl"

grpo_train_data = load_jsonl(grpo_train_file)
grpo_eval_data = load_jsonl(grpo_eval_file)

In [ ]:
# Do not change anything in this cell block

class GRPODataset(Dataset):
    """
    PyTorch Dataset for Group Relative Policy Optimization (GRPO).
    Unlike SFT or DPO, GRPO datasets only need the prompt (question) and the ground truth
    answer to evaluate the reward of the model's live generations.
    """
    def __init__(self, data: list, tokenizer, max_prompt_length: int = 512):
        """
        Initializes the GRPO dataset.

        Args:
            data (list): A list of dictionaries, where each dict contains 'question' and 'answer' keys.
            tokenizer: The Hugging Face tokenizer used for encoding the prompts.
            max_prompt_length (int): The maximum allowed sequence length for the prompt. Defaults to 512.
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_prompt_length = max_prompt_length

    def __len__(self) -> int:
        """
        Returns the total number of items in the dataset.

        Returns:
            int: The dataset size.
        """
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        """
        Retrieves and tokenizes the prompt for a specific index, preserving the raw answer for evaluation.

        This method prepares the input prompt for the model's generation phase while keeping
        the target answer in string format. This is specifically designed for RLHF workflows
        (like PPO) where the model generates a response based on 'input_ids', and the reward
        is calculated by comparing the generated text to the 'answer' string.

        Args:
            idx (int): The index of the data sample to retrieve from the dataset.

        Returns:
            dict: A dictionary containing:
                - input_ids (torch.Tensor): 1D tensor of tokenized prompt IDs,
                  truncated to `max_prompt_length`.
                - attention_mask (torch.Tensor): 1D tensor of bits indicating valid tokens.
                - answer (str): The un-tokenized ground truth response used for reward scoring.

        Example:
            >>> # Data: {"question": "What is 2+2?", "answer": "4"}
            >>> sample = dataset[0]

            >>> print(sample["input_ids"])
                torch.tensor([101, 2043, 1103, ...]) # "What is 2+2?"

            >>> print(sample["answer"])
                "4"

        Note:
            - Only the prompt is tokenized; the model is expected to generate the completion.
            - `squeeze(0)` is used to convert the tokenizer's batch output into a 1D sequence.
        """
        item = self.data[idx]
        prompt = item["question"]

        tokens = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_prompt_length,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "answer": item["answer"]  # Kept as string for exact-match reward calculation
        }


def grpo_collate_fn(batch: list, pad_token_id: int) -> dict:
    """
    Dynamically pads a batch of prompts and gathers raw answers for GRPO training.

    This function identifies the longest prompt in the current batch and right-pads
    all other sequences to match that length. Unlike standard supervised learning
    collators, it preserves the 'answers' as a list of strings, which are used
    downstream by the reward model to score the policy's generations.

    Args:
        batch (list): A list of dictionaries from GRPODataset.__getitem__,
                      each containing 'input_ids', 'attention_mask', and 'answer'.
        pad_token_id (int): The ID used for padding tokens (from the tokenizer).

    Returns:
        dict: A dictionary of batched data:
            - input_ids (torch.Tensor): 2D tensor of shape (batch_size, max_prompt_len).
            - attention_mask (torch.Tensor): 2D tensor of shape (batch_size, max_prompt_len).
            - answers (list[str]): List of raw ground truth strings.

    Example:
        >>> # Batch with prompts of lengths 4 and 6
        >>> batch = [{"input_ids": t1, "mask": m1, "answer": "4"},
        ...          {"input_ids": t2, "mask": m2, "answer": "Paris"}]

        >>> padded_batch = grpo_collate_fn(batch, pad_token_id=0)

        >>> print(padded_batch["input_ids"].shape)
            torch.Size([2, 6])

        >>> print(padded_batch["answers"])
            ["4", "Paris"]

    Note:
        - Right-padding is used to ensure the prompt tokens are aligned for the
          model's generation head.
        - The `answers` are not converted to tensors because they are used for
          string-based reward logic (e.g., regex matching or exact match).
    """
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    attention_masks = []
    answers = []

    for item in batch:
        pad = max_len - len(item["input_ids"])

        input_ids.append(F.pad(item["input_ids"], (0, pad), value=pad_token_id))
        attention_masks.append(F.pad(item["attention_mask"], (0, pad), value=0))
        answers.append(item["answer"])

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_masks),
        "answers": answers
    }

### GRPO Trainer

In [ ]:
# You can write code in some portions of this cell block, marked with BEGIN CODE and END CODE.

# ==== BEGIN EVALUATION PORTION

class GRPOTrainer:
    """
    Trainer for Group Relative Policy Optimization (GRPO).
    Samples multiple responses per prompt, calculates exact-match rewards, computes
    group-relative advantages, and updates the policy model using a PPO-clipped objective.
    """
    def __init__(self, base_model, ref_model, train_loader, eval_loader, optimizer, scheduler, device: str = None,
                 num_epochs: int = 1, group_size: int = 4, max_new_tokens: int = 256,
                 temperature: float = 1.0, clip_eps: float = 0.2, kl_beta: float = 0.02):
        """
        Initializes the GRPO Trainer.

        Args:
            base_model (BaseModel): The active policy model being trained.
            ref_model (BaseModel): The frozen reference model used to calculate KL penalties and baseline probs.
            train_loader (DataLoader): DataLoader providing the prompts.
            optimizer: PyTorch optimizer (e.g., AdamW).
            scheduler: Learning rate scheduler.
            device (str): Compute device ('cuda' or 'cpu').
            num_epochs (int): Total training epochs. Defaults to 1.
            group_size (int): Number of generations sampled per prompt (G). Defaults to 4.
            max_new_tokens (int): Maximum length of generated responses. Defaults to 256.
            temperature (float): Sampling temperature. Must be > 0 for diverse groups. Defaults to 1.0.
            clip_eps (float): PPO clipping parameter (epsilon). Defaults to 0.2.
            kl_beta (float): Weight of the KL divergence penalty. Defaults to 0.02.
        """
        self.base_model = base_model
        self.model = base_model.model.to(device)
        self.tokenizer = base_model.tokenizer

        self.ref_model = ref_model.model.to(device)
        self.ref_model.eval()
        for p in self.ref_model.parameters():
            p.requires_grad = False

        self.train_loader = train_loader
        self.eval_loader = eval_loader
        self.optimizer = optimizer
        self.scheduler = scheduler

        self.device = device

        if not device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.num_epochs = num_epochs

        self.group_size = group_size
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

        self.clip_eps = clip_eps
        self.kl_beta = kl_beta
        self.save_path = "grpo_model"

    def get_log_probs(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        Computes the average log probability of the generated completion tokens.

        This method extracts the log-likelihood of the model's own generations. It aligns
        the logits with the target labels using a causal shift, gathers the log-probabilities
        for the generated tokens, and returns the mean log-probability per sequence.
        Only the actual generated tokens (where labels != -100) are included in the average.

        Args:
            logits (torch.Tensor): Raw output scores from the policy model.
                                   Shape: (batch_size * group_size, sequence_length, vocab_size).
            labels (torch.Tensor): The generated token IDs, with prompt and padding tokens
                                   masked with -100. Shape: (batch_size * group_size, sequence_length).

        Returns:
            torch.Tensor: A 1D tensor of shape (batch_size * group_size,) representing the
                          average log probability of the completion for each group member.

        Example:
            >>> # If group_size=2 and batch_size=1
            >>> # Labels: [[-100, -100, 50, 2], [-100, -100, 60, 2]]

            >>> logps = trainer.get_logps(logits, labels)

            >>> print(logps.shape)
                torch.Size([2])
            >>> print(logps)
                torch.tensor([-0.85, -1.12])

        Note:
            - The labels must have the prompt portion masked with -100 to ensure the
              reward/advantage is only calculated over the generated response.
            - Shifting is performed internally: logit at index `t` is matched with label at `t+1`.
            - We average by the number of valid tokens to prevent the model from being
              penalized simply for generating longer, detailed responses.
        """
        # BEGIN CODE : grpo_trainer.get_log_probs
        # ADD YOUR CODE HERE

        # END CODE

    def extract_answer(self, text: str):
        """
        Extracts the final numerical answer from the generation using Regex.

        Args:
            text (str): The raw generated text string.

        Returns:
            str or None: The extracted numerical answer as a string, or None if no number was found.
        """
        patterns = [
            r"####*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
            r"\\boxed\{\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)\s*\}",
            r"[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
            r"[Aa]nswer\s*[:\-]?\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
            r"[Tt]he answer is\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)",
            r"=\s*(-?\d+(?:\.\d+)?(?:e[-+]?\d+)?)\s*$"
        ]

        for pattern in patterns:
            match = re.search(pattern, text)
            if match:
                return match.group(1)

        numbers = re.findall(r"-?\d+(?:\.\d+)?(?:e[-+]?\d+)?", text)
        if numbers:
            return numbers[-1]

        return None

    def compute_rewards(self, responses: list, answers: list) -> torch.Tensor:
        """
        Calculates a binary reward based on an exact string match between the extracted prediction
        and the ground truth answer.

        This method implements a sparse, outcome-based reward function. It parses the model's
        raw text output to isolate the final answer and compares it against the gold standard.
        In the context of GRPO, these rewards are used to compute relative advantages
        within each group of generations.

        Args:
            responses (list): A list of raw strings generated by the model.
                              Length: (batch_size * group_size).
            answers (list): A list of ground truth strings. These are repeated to align
                            with the expanded group size. Length: (batch_size * group_size).

        Returns:
            torch.Tensor: A 1D FloatTensor where each element is 1.0 for a correct match
                          or 0.0 for an incorrect or unparseable response.

        Example:
            >>> # batch_size=1, group_size=3, ground_truth="42"
            >>> responses = ["The answer is 42", "It is 41", "I don't know"]
            >>> answers = ["42", "42", "42"]

            >>> rewards = trainer.compute_rewards(responses, answers)

            >>> print(rewards)
                torch.tensor([1.0, 0.0, 0.0])

        Note:
            - The `extract_answer` helper is crucial for stripping conversational filler
              from the model's response before comparison.
            - If the extraction logic fails to find a numerical candidate, the reward
              defaults to 0.0.
            - Both prediction and ground truth are cast to strings to ensure
              consistent comparison.
        """
        # BEGIN CODE : grpo_trainer.compute_rewards
        # ADD YOUR CODE HERE

        # END CODE

    def sample_responses(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Duplicates the prompts `group_size` times and generates responses using sampling.

        Args:
            input_ids (torch.Tensor): Prompt token IDs of shape (batch_size, seq_len).
            attention_mask (torch.Tensor): Prompt attention mask of shape (batch_size, seq_len).

        Returns:
            torch.Tensor: The generated token IDs (including the prompts) of shape
                          (batch_size * group_size, combined_seq_len).
        """
        # Duplicate prompts for group sampling
        input_ids = input_ids.repeat_interleave(self.group_size, dim=0)
        attention_mask = attention_mask.repeat_interleave(self.group_size, dim=0)

        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=self.max_new_tokens,
            do_sample=True, # Must be true to get diverse responses within the group
            temperature=self.temperature,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        return outputs

    def compute_advantages(self, rewards: torch.Tensor) -> torch.Tensor:
        """
        Computes the normalized advantage for each generation strictly within its prompt group.

        In Group Relative Policy Optimization (GRPO), advantages are calculated by
        comparing the reward of a specific generation against the average performance
        of other completions from the same prompt. This removes the need for a separate
        critic (Value) model, as the 'baseline' is derived dynamically from the
        group's mean and standard deviation.

        Args:
            rewards (torch.Tensor): 1D tensor of raw rewards of shape (batch_size * group_size,).

        Returns:
            torch.Tensor: 1D tensor of shape (batch_size * group_size,) representing
                          the Z-scored advantages within each prompt group.

        Example:
            >>> # batch_size=1, group_size=3
            >>> # Rewards: [1.0, 0.0, 0.0]
            >>> # Mean: 0.33, Std: 0.57

            >>> advantages = trainer.compute_advantages(torch.tensor([1.0, 0.0, 0.0]))

            >>> print(advantages)
                torch.tensor([ 1.1547, -0.5774, -0.5774])

        Note:
            - The method reshapes the input to process multiple prompt groups
              simultaneously in a vectorized manner.
            - An epsilon (like 1e-8) is added to the standard deviation to ensure numerical
              stability if all completions in a group receive the same reward.
            - Positive advantages indicate completions that outperformed the group average.
        """
        # BEGIN CODE : grpo_trainer.compute_advantages
        # ADD YOUR CODE HERE

        # END CODE

    def compute_loss(self, batch: dict) -> torch.Tensor:
        """
        Executes the full GRPO training step: generating responses, computing rewards,
        normalizing advantages, and calculating the clipped surrogate objective.

        This method implements the core GRPO logic, which optimizes a policy model by
        sampling multiple completions per prompt. It uses Group Relative advantages
        to determine which completions were better than average and applies a
        PPO-style clipping mechanism and KL-penalty to ensure training stability.

        Args:
            batch (dict): A dictionary containing:
                - "input_ids": Tokenized prompts of shape (batch_size, seq_len).
                - "attention_mask": Attention mask for the prompts.
                - "answers": A list of ground truth answer strings of length (batch_size).

        Returns:
            torch.Tensor: A scalar float tensor representing the combined loss
                          (policy loss + KL penalty) to be backpropagated.

        Example:
            >>> # Process a batch through the trainer
            >>> loss = grpo_trainer.compute_loss(batch)
            >>> loss.backward()
            >>> print(loss.item())
                0.1420

        Note:
            - Generation: The `sample_responses` method expands the batch by `group_size`.
            - Dynamic Masking: A custom `gen_attention_mask` is generated to prevent the
              model from attending to trailing padding tokens during the forward pass.
            - Label Masking: Prompt tokens and padding tokens are explicitly masked with
              -100 in the `labels` tensor so `get_log_probs` only evaluates the newly
              generated text.
            - Importance Sampling: Uses the ratio between current policy and reference
              model probabilities to allow for safe policy updates.
            - Clipping: The `clip_eps` parameter prevents the policy from changing too
              drastically in a single step based on extreme reward values.
            - KL Penalty: A regularization term that prevents the model from "gaming"
               the reward function by staying mathematically close to the reference model.
        """
        # BEGIN CODE : grpo_trainer.compute_loss
        # ADD YOUR CODE HERE

        # END CODE

    def train_epoch(self) -> float:
        """
        Executes a single epoch of GRPO training over the dataloader.

        Returns:
            float: The average loss for the epoch.
        """
        self.model.train()
        total_loss = 0

        for batch in tqdm(self.train_loader, desc="GRPO Training"):
            loss = self.compute_loss(batch)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def eval_epoch(self):
        """
        Runs a single epoch of evaluation without backpropagation.
        """
        self.model.eval()
        total_loss = 0

        with torch.no_grad():
            for batch in tqdm(self.eval_loader, desc="GRPO Evaluating"):
                loss = self.compute_loss(batch)
                total_loss += loss.item()

        avg_loss = total_loss / len(self.eval_loader)

        return avg_loss

    def train(self):
        """
        Executes the full training loop across the specified number of epochs.

        Returns:
            None
        """
        for epoch in range(self.num_epochs):
            train_loss = self.train_epoch()
            eval_loss = self.eval_epoch()

            print(
                f"Epoch {epoch+1}/{self.num_epochs} "
                f"- Train Loss: {train_loss:.4f} "
                f"- Eval Loss: {eval_loss:.4f}"
            )
        self.base_model.save(self.save_path)
# ==== END EVALUATION PORTION

### Parameters and Config for Training

In [ ]:
# ==== BEGIN EVALUATION PORTION

# BEGIN CODE : grpo_params
# ADD YOUR CODE HERE
# Default Parameters are given here. You can keep them as is, but you can also experiment if you wish to.

GRPO_MAX_PROMPT_LENGTH = 256
GRPO_BATCH_SIZE = 4
GRPO_GROUP_SIZE = 4

GRPO_MAX_NEW_TOKENS = 256
GRPO_TEMPERATURE = 0.7

GRPO_LEARNING_RATE = 2e-7
GRPO_WEIGHT_DECAY = 0.01

GRPO_CLIP_EPS = 0.2
GRPO_KL_BETA = 0.02

GRPO_NUM_EPOCHS = 1

# END CODE

# ==== END EVALUATION PORTION

policy_model = BaseModel("dpo_model")

ref_model = BaseModel("dpo_model")

grpo_train_dataset = GRPODataset(
    grpo_train_data,
    policy_model.tokenizer,
    max_prompt_length=GRPO_MAX_PROMPT_LENGTH
)

grpo_eval_dataset = GRPODataset(
    grpo_eval_data,
    policy_model.tokenizer,
    max_prompt_length=GRPO_MAX_PROMPT_LENGTH
)

GRPO_COLLATE_FN = lambda batch: grpo_collate_fn(
    batch,
    policy_model.tokenizer.pad_token_id
)


grpo_train_loader = DataLoader(
    grpo_train_dataset,
    batch_size=GRPO_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    collate_fn=GRPO_COLLATE_FN
)


grpo_eval_loader = DataLoader(
    grpo_eval_dataset,
    batch_size=GRPO_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    collate_fn=GRPO_COLLATE_FN
)


grpo_optimizer = torch.optim.AdamW(
    policy_model.model.parameters(),
    lr=GRPO_LEARNING_RATE,
    weight_decay=GRPO_WEIGHT_DECAY
)


grpo_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    grpo_optimizer,
    T_max=GRPO_NUM_EPOCHS * len(grpo_train_loader)
)

In [ ]:
grpo_trainer = GRPOTrainer(
    base_model=policy_model,
    ref_model=ref_model,
    train_loader=grpo_train_loader,
    eval_loader=grpo_eval_loader,
    optimizer=grpo_optimizer,
    scheduler=grpo_scheduler,
    num_epochs=GRPO_NUM_EPOCHS,
    group_size=GRPO_GROUP_SIZE,
    max_new_tokens=GRPO_MAX_NEW_TOKENS,
    temperature=GRPO_TEMPERATURE,
    clip_eps=GRPO_CLIP_EPS,
    kl_beta=GRPO_KL_BETA,
)

grpo_trainer.train()

### Evaluation

In [ ]:
# Load Models

grpo_model = BaseModel("grpo_model")

In [ ]:
# Sample GRPO Model Generations

print("Sample GRPO Model Generations\n")

for data in sample_data:
    pred = grpo_model.generate(
        data['question'],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY
    )
    print(f"Question:\n {data['question']}\n")
    print(f"Output:\n {pred}")
    print("---"*20)
    print("\n")

In [ ]:
# Pass@1

grpo_pass1 = pass_at_1(
    grpo_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print("\n\n")
print(f"Base Model Pass@1: {base_pass1}")
print(f"SFT Model Pass@1: {sft_pass1}")
print(f"DPO Model Pass@1: {dpo_pass1}")
print(f"GRPO Model Pass@1: {grpo_pass1}")


In [ ]:
# Pass@K

grpo_passK = pass_at_k(
    grpo_model,
    eval_data,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

print(f"Base Model Pass@K: {base_passK}")
print(f"SFT Model Pass@K: {sft_passK}")
print(f"DPO Model Pass@K: {dpo_passK}")
print(f"GRPO Model Pass@K: {grpo_passK}")

In [ ]:
# Clear memory
flush_memory(policy_model, ref_model, grpo_trainer, grpo_model, grpo_optimizer, grpo_scheduler)

del policy_model, ref_model, grpo_trainer, grpo_model, grpo_optimizer, grpo_scheduler

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Zipping Model Folders

You can use the below utility code to zip the model files, so that you only have to download the corresponding zip files from Colab.

In [ ]:
# Zip the SFT Model Files
shutil.make_archive("sft_model", "zip", "sft_model")

In [ ]:
# Zip the DPO Model Files
shutil.make_archive("dpo_model", "zip", "dpo_model")

In [ ]:
# Zip the GRPO Model Files [Run this cell only if you trained the GRPO model]
shutil.make_archive("grpo_model", "zip", "grpo_model")

## Summary and Takeaways

1. Start with a pretrained model. Even a tiny model with prior exposure to
   mathematical text is far better than training from scratch. GPT-2 already knows
   what numbers and arithmetic operations look like.

2. SFT is the foundation. No amount of preference learning or RL can help
   if the model doesn't first know the format it's supposed to follow.
   Chain-of-thought SFT teaches the model to "show its work".

3. DPO is like a surgical refinement. It nudges the model toward correct reasoning
   without destroying the SFT foundation. The β parameter is the key dial,
   too high and nothing changes; too low and the model forgets SFT.

4. GRPO is like self-study. It gives the model the ability to learn from its own
   attempts, not just from labelled examples.
   In principle, given enough compute, a model can keep improving through GRPO
   without any new human-labelled data.

5. The metrics matter. Pass@1 is what a real student experiences.
   Pass@K shows the model's upper bound. Tracking both reveals whether the
   model is getting smarter or just getting luckier.

**Note on Evaluation Metrics**

In this assignment, our Pass@1 and Pass@k functions use a permissive extraction logic. We search for multiple patterns (e.g., `####`, `\boxed{}`) and, as a final fallback, extract the last numerical value found in the model's generation.

While this generous approach helps credit models that reach the correct result despite formatting errors, it is not standard for formal benchmarking. In rigorous research settings, evaluation is much stricter.

- For GSM8K, models are typically required to strictly follow the dataset's format, where the final answer must immediately follow the four-hash delimiter (####).

- Relying on the last number can lead to "False Positives" if a model mentions a date or a constant (e.g., "as of 2026") after its actual calculation.

- To report reproducible metrics in a paper or project, we recommend using standardized frameworks like the [EleutherAI LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness), which uses fixed regex filters and multi-shot Chain-of-Thought (CoT) prompting to ensure fair comparison across models.

## Completion Checklist

These are the sections where you can edit code/parameters.

- Base Model and Evaluation Functions
    - Generation Parameters
        - Can edit parameters
- SFT
  - SFT Dataset
    - `SFTDataset._format_samples()`
    - `SFTDataset._apply_causal_mask()`
  - SFT Trainer
    - `SFTTrainer._shift_cross_entropy()`
  - Parameters and Config for Training
    - Can edit parameters
- DPO
  - DPO Dataset
    - `DPODataset._tokenize_pair()`
  - DPO Trainer
    - `DPOTrainer._get_log_probs()`
    - `DPOTrainer._compute_implicit_rewards()`
    - `DPOTrainer._compute_preference_loss()`
    - `DPOTrainer._compute_loss()`
  - Parameters and Config for Training
    - Can edit parameters
- GRPO [Optional Section]
  - GRPO Trainer
    - `GRPOTrainer.get_log_probs()`
    - `GRPOTrainer.compute_rewards()`
    - `GRPOTrainer.compute_advantages()`
    - `GRPOTrainer.compute_loss()`
  - Parameters and Config for Training
    - Can edit parameters

## TA Anecdotes

Thanks for making it to the end. This assignment was created from scratch, after many days of grueling effort and experimentation. Feel free to read on these stories from our time creating this assignment, or skip them.

Right from the planning stage, we were constrained by the requirement that the notebook should run on Colab. Within Colab's 16 GB memory. Within Colab's free time limit. That reduces the choices of model and datasets we can use by... a lot.

We had an ambitious plan. Have you ever seen adventure games where the first quest is to fetch a kitten, and the last quest is to kill god? Well, this assignment was almost going to become like that, with Victor setting up a full pre-training an LLM (or well, Small Language Model - SLM) of like 30M parameters. It was pushed back, for good reason, by Danish and Aditya. We stuck to RL methods after that.

We tried many, many different models, in hope of finding a half-decent one that can train fine under the constraints and also show improvement, even if marginally. You might be used to seeing GSM-8K benchmarks in 80%+ range, but getting a tiny model of even a few hundred million parameters to do math is a lot more challenging than it looks. At the scale of tiny models that we are operating at in this notebook, even a few percentage points is actually pretty cool.

Victor's pre-training setup was not completely in vain, or free of incidents. During pre-training our own small language model *from scratch* to learn language and do math on an H200 GPU on a server for over a few hours, Victor received a mail from Neetha Ma'am, the System Administrator, asking about what are we even doing on this GPU, and that its temperatures were too high for a while.

Are you even using a server to its full extent if you don't get a mail from sysadmin asking you to stop? Jokes apart, please be responsible with your server usage.

This assignment notebook has seen some rough days, and probably will see some more as we discover more errors and things to be fixed, and things to be clarified. We thank you for being patient with it till the end.

\- Purva and Victor